# 결선 핵심 분석 — 같은 동네, 다른 시간
3-2(같은 OD 모형, 시간대별 계수) · 3-3(파업) · 3-5b(공간-시간 불일치) · Phase 4(탄소, CNG 민감도) · P2 재배분. 프로젝트 루트에서 실행.

심사 반영(2026-10-22): 탄소 헤드라인은 시간대별 계수 기준 심야·새벽 값이다. 시간대 공통 계수 값은 상한 시나리오로만 보여 준다.

In [1]:
import os, json, pandas as pd, numpy as np, pyfixest as pf
os.chdir('..') if os.path.basename(os.getcwd()) == 'notebooks' else None
pan = pd.read_parquet('data/interim/02_패널.parquet'); pan['od'] = pan.o + '_' + pan.d
print(pan.shape, pan.od.nunique())

(1277629, 25) 87574


## 3-2. 같은 길, 다른 시간 — OD 고정효과 + 시간구간×요일 고정효과

In [2]:
m = pf.feols('car_share ~ S_o + S_d | od + tbin^daytype', data=pan, weights='trips', vcov={'CRV1': 'o'})
m.summary()

###

Estimation:  OLS
Dep. var.: car_share, Fixed effects: od + tbin^daytype
sample: None = all
Inference:  CRV1
Observations:  1277629

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| S_o           |     -0.034 |        0.009 |    -3.919 |      0.000 | -0.051 |  -0.017 |
| S_d           |     -0.014 |        0.005 |    -2.803 |      0.005 | -0.024 |  -0.004 |
---
RMSE: 0.501 R2: 0.621 R2 Within: 0.0 


In [3]:
# 낮 구간끼리만(05–22) — 식별 출처 점검
m3 = pf.feols('car_share ~ S_o + S_d | od + tbin^daytype', data=pan[~pan.tbin.isin(['22-24','00-05'])], weights='trips', vcov={'CRV1': 'o'})
m3.tidy()[['Estimate','Std. Error','Pr(>|t|)']]

,Estimate,Std. Error,Pr(>|t|)
Coefficient,,,
S_o,-0.001230,0.014500,0.932414
S_d,0.010662,0.004925,0.030955


## 3-5b. 공간-시간 불일치 2×2

In [4]:
c = pd.read_parquet('outputs/03_셀.parquet')
d = c.groupby('dong').agg(공간양호=('space_good','first'), 시간사각=('blind','any'))
pd.crosstab(d.공간양호, d.시간사각, margins=True)

시간사각,False,True,All
공간양호,,,
0,21,49,70
1,262,95,357
All,283,144,427


## 3-2 보강. 시간대별 계수 — 그 시간대 안의 변동만(OD×시간대 고정효과)

In [5]:
pan['band'] = np.select([pan.tbin.isin(['22-24','00-05']), pan.tbin=='05-07'], ['night','dawn'], 'day')
for b in ['night','dawn','day']:
    pan[f'So_{b}'] = pan.S_o*(pan.band==b); pan[f'Sd_{b}'] = pan.S_d*(pan.band==b)
pan['odb'] = pan.od + '_' + pan.band
mb = pf.feols('car_share ~ So_night + Sd_night + So_dawn + Sd_dawn + So_day + Sd_day | odb + tbin^daytype', data=pan, weights='trips', vcov={'CRV1': 'o'})
mb.tidy()[['Estimate','Std. Error','Pr(>|t|)','2.5%','97.5%']].round(4)

C:\Users\<user>\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 25995 singleton fixed effect(s) dropped from the model.
  warnings.warn(


,Estimate,Std. Error,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,
So_night,-0.0799,0.0185,0.0000,-0.1163,-0.0436
Sd_night,-0.0049,0.0089,0.5827,-0.0224,0.0126
So_dawn,0.0466,0.0419,0.2674,-0.0358,0.1290
Sd_dawn,-0.2593,0.0283,0.0000,-0.3148,-0.2037
So_day,0.0228,0.0202,0.2590,-0.0169,0.0626
Sd_day,0.0318,0.0064,0.0000,0.0192,0.0444


## 3-3. 파업: 차량 '비중'은 올랐지만 차량 '통행 수'는 늘지 않았다(낮 07–22시)

In [6]:
print(pd.read_csv('outputs/03_파업_시간대별.csv').round(4).to_string())
pd.read_csv('outputs/03_파업_차량통행.csv').round(4)

             band          y    coef      se       p      lo      hi      n
0        주간 07–22  car_share  0.0512  0.0091  0.0000  0.0333  0.0691  12810
1        주간 07–22  log_trips -0.1589  0.0186  0.0000 -0.1954 -0.1223  12810
2  평일 낮 비첨두 09–16  car_share  0.0427  0.0105  0.0001  0.0221  0.0632   5124
3  평일 낮 비첨두 09–16  log_trips -0.1770  0.0248  0.0000 -0.2256 -0.1283   5124
4  첨두 07–09·16–19  car_share  0.0400  0.0103  0.0001  0.0197  0.0603   5124
5  첨두 07–09·16–19  log_trips -0.1229  0.0201  0.0000 -0.1624 -0.0833   5124
6     심야·새벽 22–07  car_share  0.0613  0.0171  0.0004  0.0277  0.0949   7686
7     심야·새벽 22–07  log_trips -0.0347  0.0370  0.3483 -0.1074  0.0380   7686


,band,y,coef,se,p,lo,hi,n
0,전체,log_car,-0.0041,0.0366,0.9108,-0.0761,0.0679,20496
1,주간 07–22,log_car,-0.0134,0.0376,0.7216,-0.0873,0.0605,12810
2,평일 낮 비첨두 09–16,log_car,-0.0910,0.0417,0.0297,-0.1730,-0.0090,5124
3,첨두 07–09·16–19,log_car,0.0041,0.0449,0.9265,-0.0840,0.0923,5124
4,심야·새벽 22–07,log_car,0.0588,0.0595,0.3236,-0.0582,0.1758,7686


## Phase 4. 연관 추가 배출 — 시간대별 계수(헤드라인: 심야·새벽) vs 시간대 공통 계수(상한 시나리오)

In [7]:
cb = json.load(open('outputs/04c_시간대별_탄소.json', encoding='utf-8'))
for k in ['band_nd','band_night','band_dawn','band_day']:
    print(k, {kk: round(v) for kk, v in cb[k]['기본'].items()})
print('공통 계수 상한(기본):', round(cb['common_beta_upper_t']))
pd.read_csv('outputs/04c_구간별_배출.csv').pivot(index='tbin', columns='daytype', values='tco2_ann').round(0)

band_nd {'t': 6679, 'ci_lo': 2120, 'ci_hi': 11289}
band_night {'t': 3207, 'ci_lo': 1933, 'ci_hi': 4593}
band_dawn {'t': 3472, 'ci_lo': -584, 'ci_hi': 7438}
band_day {'t': -27942, 'ci_lo': -48596, 'ci_hi': -9401}
공통 계수 상한(기본): 28124


daytype,we,wk
tbin,,
00-05,672.0,1364.0
05-07,738.0,2734.0
07-09,-708.0,-3791.0
09-12,-1564.0,-3048.0
12-16,-2430.0,-4438.0
16-19,-1709.0,-5509.0
19-22,-1295.0,-3451.0
22-24,329.0,842.0


## RQ4. CNG 증회 — 승용차 계수를 올려도 결론이 유지되는가

In [8]:
pd.read_csv('outputs/04c_CNG_민감도.csv')

,beta,car_mult,car_g_km,occupancy,pos_cells,cells,net_t_all
0,common,1.0,125.2,기본 1.4,0,1312,-186471.773276
1,common,1.0,125.2,낙관 1.2,0,1312,-184690.570203
2,common,1.5,187.8,기본 1.4,0,1312,-181128.164057
3,common,1.5,187.8,낙관 1.2,0,1312,-178456.359448
4,common,2.0,250.4,기본 1.4,0,1312,-175784.554838
5,common,2.0,250.4,낙관 1.2,0,1312,-172222.148692
6,common,3.0,375.6,기본 1.4,1,1312,-165097.336400
7,common,3.0,375.6,낙관 1.2,1,1312,-159753.727181
8,band,1.0,125.2,기본 1.4,0,1312,-195429.476386
9,band,1.0,125.2,낙관 1.2,0,1312,-195141.223832


## P2. 운행 총량 그대로 재배분(시간대별 계수, 첨두 운행 유지) — 시뮬레이션

In [9]:
r5 = json.load(open('outputs/05_재배분_시간대_첨두제외.json', encoding='utf-8'))
print({k: v for k, v in r5.items() if k.startswith('topN') or k in ('cells_with_move','neg_cells','all_net_t')})
pd.read_csv('outputs/05_재배분_시간대_첨두제외.csv').head(10)[['adm_nm','daytype','tbin','space_good','old_share','moved_runs_yr','net_t','net_kg_per_run']].round(3)

{'cells_with_move': 264, 'neg_cells': 0, 'all_net_t': {'보수': 1197.285258539969, '기본': 1459.5477437439622, '낙관': 1702.8057010346226}, 'topN': 20, 'topN_net_t': {'보수': 420.1438075532206, '기본': 512.1753082553547, '낙관': 597.5378596312471}, 'topN_moved_runs_yr': 968494.8056933774, 'topN_kg_per_run': {'보수': 0.4338111108943179, '기본': 0.5288364018521209, '낙관': 0.6169758021608077}, 'topN_nonnight': 0.75, 'topN_space_good': 0.7, 'topN_tbins': {'we|00-05': 1, 'wk|00-05': 4, 'wk|05-07': 15}}


,adm_nm,daytype,tbin,space_good,old_share,moved_runs_yr,net_t,net_kg_per_run
0,서울특별시 종로구 청운효자동,wk,05-07,0,0.217,60686.928,68.518,1.129
1,서울특별시 양천구 목5동,wk,05-07,1,0.152,55164.548,55.836,1.012
2,서울특별시 성동구 성수2가1동,wk,05-07,0,0.251,53593.167,48.369,0.903
3,서울특별시 서초구 반포3동,wk,05-07,1,0.161,51090.805,42.568,0.833
4,서울특별시 동대문구 회기동,wk,05-07,1,0.159,54153.118,33.127,0.612
5,서울특별시 강서구 방화3동,wk,05-07,1,0.305,29592.263,23.943,0.809
6,서울특별시 성북구 월곡2동,wk,05-07,1,0.235,52399.863,23.921,0.457
7,서울특별시 영등포구 대림3동,wk,05-07,1,0.271,51935.318,23.321,0.449
8,서울특별시 종로구 평창동,wk,05-07,0,0.249,54047.148,22.191,0.411
9,서울특별시 마포구 망원1동,wk,05-07,0,0.211,69525.475,18.503,0.266
